# Day 5.4 — Permissions, Approval and Limits
The model proposes; Python disposes. Here that becomes running code: a risk table that fails closed,
a run that *pauses* for a human instead of asking the model, and a step limit no configuration can
raise.

### The rules being encoded

A decision uses two facts: is the tool allow-listed, and what is its local **risk level**?

| Risk | Meaning | Decision |
|---|---|---|
| `read` | no effect outside the process | allow |
| `write` | reversible local change | allow |
| `external` | leaves the machine or is visible to others | approval |
| `destructive` | irreversible | deny |

Anything not in that table returns `deny` — **failing closed**. Denying beats raising: a crash can be
caught and ignored, a `deny` gets logged.

`approval` is not asked in the conversation. The run stops, the pending action goes into a
**checkpoint**, and a separate `resume` call carries the human answer. Rejection is a *successful*
outcome with its own status, `cancelled`. Finally, a configuration may request any number of steps,
but the runtime owns the ceiling.

In [ ]:
# The one place risk levels turn into permission decisions.
RISK_POLICY = {
    "read": "allow",          # no side effect outside the process
    "write": "allow",         # reversible local change
    "external": "approval",   # leaves the machine or is visible to others -> pause for a human
    "destructive": "deny",    # never automatic in this course
}
UNKNOWN_RISK_DECISION = "deny"          # returned for any label we do not recognise

def decide(config, spec):
    """Return 'allow', 'approval' or 'deny' for this agent calling this tool."""
    if spec.name not in config.allowed_tools:
        return "deny"                                   # not on the allow-list, whatever its risk
    # .get(...) rather than [...]: an unknown risk label denies instead of raising KeyError.
    return RISK_POLICY.get(spec.risk, UNKNOWN_RISK_DECISION)

print("risk level  -> decision")
for risk, outcome in RISK_POLICY.items():
    print(f"  {risk:<12}-> {outcome}")
print()
print("For task_agent, tool by tool:")
for spec in registry.discover():
    print(f"  {spec.name:<17} risk={spec.risk:<12} allow-listed={str(spec.name in task.allowed_tools):<5} "
          f"-> {decide(task, spec)}")
print()
print("create_draft and send_email are BOTH allow-listed and get different answers.")

### Step 1 — The runtime: one loop every agent shares

Read it from the top: scoped discovery, the bounded provider call, validation, policy, the approval
pause and the step limit each appear once.

In [ ]:
from uuid import uuid4

MAX_STEPS_HARD_CAP = 10     # the runtime's ceiling; a configuration cannot raise it

def effective_step_limit(config):
    """The number of steps this run may actually take."""
    return min(config.max_steps, MAX_STEPS_HARD_CAP)

class CheckpointStore:
    """Mutable continuation state for a paused run (5.5 makes it durable)."""
    def __init__(self): self.items = {}
    def save(self, run_id, state): self.items[run_id] = state
    def load(self, run_id): return self.items.get(run_id)
    def delete(self, run_id): self.items.pop(run_id, None)

class HarnessRuntime:
    def __init__(self, registry, provider, events=None, checkpoints=None):
        self.registry, self.provider = registry, provider
        self.events = events or EventStore()
        self.checkpoints = checkpoints or CheckpointStore()

    def run(self, config, prompt, run_id=None):
        run_id = run_id or uuid4().hex[:8]
        history, limit = [], effective_step_limit(config)
        self.events.add(run_id, "run_started", agent=config.name, model=config.model.model,
                        requested_max_steps=config.max_steps, hard_cap=MAX_STEPS_HARD_CAP,
                        effective_step_limit=limit)
        for step in range(1, limit + 1):                                  # bounded: never forever
            tools = self.registry.discover(config.allowed_tools)          # scoped discovery
            self.events.add(run_id, "model_requested", step=step, visible_tools=[t.name for t in tools])
            try:
                choice = call_provider_with_retries(self.provider, prompt, config, tools,
                                                    history, self.events, run_id, step)
            except Exception as exc:
                return self._failed(run_id, f"Model error: {exc}")
            self.events.add(run_id, "model_completed", step=step, usage=choice.usage)
            if choice.kind == "final":
                self.events.add(run_id, "run_completed", step=step)
                return RunResult(run_id, "completed", choice.content, events=self.events.get(run_id))
            try:
                spec = self.registry.get(choice.tool or "").spec          # unknown tool -> KeyError
            except KeyError as exc:
                return self._failed(run_id, str(exc))
            try:
                self.registry.validate(spec.name, choice.arguments)       # shape before policy
            except (ValueError, TypeError) as exc:
                return self._failed(run_id, str(exc))
            decision = decide(config, spec)                               # PYTHON decides
            self.events.add(run_id, "policy_decision", tool=spec.name, risk=spec.risk, decision=decision)
            history.append({"role": "assistant", "tool": spec.name, "arguments": choice.arguments})
            if decision == "deny":
                return self._failed(run_id, f"Denied tool: {spec.name}")
            if decision == "approval":                                    # stop, and write it down
                pending = {"tool": spec.name, "arguments": choice.arguments, "prompt": prompt,
                           "agent": config.name, "steps_used": step, "history": history}
                self.checkpoints.save(run_id, pending)
                self.events.add(run_id, "approval_requested", tool=spec.name, arguments=choice.arguments)
                return RunResult(run_id, "pending_approval", pending_action=pending,
                                 events=self.events.get(run_id))
            try:
                output = self.registry.call(spec.name, choice.arguments, config.allowed_tools)
            except Exception as exc:      # a tool refusing (e.g. a guardrail) is a recorded outcome
                self.events.add(run_id, "tool_failed", tool=spec.name, error=str(exc))
                return self._failed(run_id, f"Tool {spec.name} refused: {exc}")
            history.append({"role": "tool", "name": spec.name, "content": str(output)})
            self.events.add(run_id, "tool_completed", tool=spec.name)
        self.events.add(run_id, "step_limit_reached", limit=limit,
                        requested_max_steps=config.max_steps, hard_cap=MAX_STEPS_HARD_CAP)
        return RunResult(run_id, "step_limit", f"Maximum steps reached (effective limit {limit}).",
                         events=self.events.get(run_id))

    def resume(self, run_id, config, approved):
        """Carry a human answer back into a paused run. The model is NOT asked again."""
        state = self.checkpoints.load(run_id)
        if not state:
            return self._failed(run_id, "Checkpoint not found")
        self.events.add(run_id, "approval_resolved", approved=approved)
        self.checkpoints.delete(run_id)                    # consumed: it cannot be approved twice
        if not approved:
            self.events.add(run_id, "run_cancelled", reason="A person rejected the action")
            return RunResult(run_id, "cancelled", "A person rejected the action",
                             events=self.events.get(run_id))
        spec = self.registry.get(state["tool"]).spec
        if decide(config, spec) != "approval":             # the world may have changed while paused
            return self._failed(run_id, "Policy changed while the run was paused")
        try:
            output = self.registry.call(spec.name, state["arguments"], config.allowed_tools)
        except Exception as exc:
            self.events.add(run_id, "tool_failed", tool=spec.name, error=str(exc))
            return self._failed(run_id, f"Tool {spec.name} refused: {exc}")
        self.events.add(run_id, "tool_completed", tool=spec.name)
        self.events.add(run_id, "run_completed", resumed=True)
        return RunResult(run_id, "completed", str(output), events=self.events.get(run_id))

    def _failed(self, run_id, message):
        self.events.add(run_id, "run_failed", error=message)
        return RunResult(run_id, "failed", message, events=self.events.get(run_id))

runtime = HarnessRuntime(build_demo_registry(), MockProvider())
first = runtime.run(research, "harness")
print("Runtime ready. Hard step cap:", MAX_STEPS_HARD_CAP)
print("research_agent, one run ->", first.status, "|", first.output[:90])
print("Trace:", [e["event"] for e in first.events])
print("Every run ends in exactly one of: completed | pending_approval | cancelled | failed | step_limit")

### Step 2 — An external action pauses the run

`send_email` is classified `external`. The run does not fail and does not ask the model. It stops,
and writes down exactly what it wanted to do.

In [ ]:
pending = runtime.run(task, "Send a synthetic course update")

print("Status      :", pending.status)
print("Paused tool :", pending.pending_action["tool"])
print("Exact arguments a human is being asked to approve:")
for key, value in pending.pending_action["arguments"].items():
    print(f"    {key}: {value}")
print()
print("Trace       :", [e["event"] for e in pending.events])
print("Checkpoint  : run", pending.run_id, "keys", sorted(runtime.checkpoints.load(pending.run_id)))

### Step 3 — Rejecting is a success, and it has its own status

A rejected run is `cancelled`, not `failed`. Nothing went wrong; a person said no.

In [ ]:
rejected = runtime.resume(pending.run_id, task, approved=False)

print("Status after rejection:", rejected.status)
print("Is that a failure?    :", rejected.status == "failed")
print("Message               :", rejected.output)
print("Last three events     :", [e["event"] for e in rejected.events[-3:]])
print("Checkpoint afterwards :", runtime.checkpoints.load(pending.run_id), "- cleared, so it cannot be replayed")

### Step 4 — Two refusals nobody had to ask about

Allow-listing a `destructive` tool changes nothing, and an unseen risk label denies rather than
raising — and 5.6 imports a tool from a server you do not control.

In [ ]:
# 1) Over-permissive configuration: erase_workspace really is on the allow-list,
#    and the mock plan really does request it.
greedy = load_config("task_agent")
greedy.allowed_tools.append("erase_workspace")
greedy.mock_plan = [{"tool": "erase_workspace", "arguments": {}}]

destructive = registry.get("erase_workspace").spec
print("On the allow-list :", "erase_workspace" in greedy.allowed_tools)
print("Therefore visible :", "erase_workspace" in [s.name for s in registry.discover(greedy.allowed_tools)])
print("Policy decision   :", decide(greedy, destructive))
outcome = runtime.run(greedy, "Erase everything")
print("A run that asks for it ->", outcome.status, "|", outcome.output)
print("Events            :", [e["event"] for e in outcome.events])
print("The tool function never ran, and the denial is itself an event.")
print()

# 2) A tool whose risk label is not one of read/write/external/destructive.
mystery = ToolSpec("mystery_action", "A capability from somewhere else",
                   {"type": "object", "properties": {}}, "quantum")
somebody = AgentConfig("somebody", "Try anything.", ["mystery_action"])
print("Is 'quantum' in the policy table?", "quantum" in RISK_POLICY)
print("Decision for an unknown risk    :", decide(somebody, mystery), "<- fails closed, does not raise")

### Step 5 — Limits the configuration cannot raise

A model that never says "done", and the two numbers that stop it.

In [ ]:
class EndlessModel:
    """Always asks for the same tool. Never says 'done'."""
    def decide(self, prompt, config, tools, history):
        return ModelDecision("tool", tool="lookup_notes", arguments={"query": prompt})

print("Hard cap written into the runtime:", MAX_STEPS_HARD_CAP)
for requested in (2, 500):
    greedy_config = load_config("research_agent")
    greedy_config.max_steps = requested
    print(f"  config asks for {requested:>3} steps -> effective limit {effective_step_limit(greedy_config)}")

looping = load_config("research_agent")
looping.max_steps = 500
stopped = HarnessRuntime(build_demo_registry(), EndlessModel()).run(looping, "keep going")
print()
print("Status     :", stopped.status)
print("First event:", stopped.events[0]["event"], stopped.events[0]["details"])
print("Last event :", stopped.events[-1]["event"], stopped.events[-1]["details"])
print()
print("The event reports the EFFECTIVE limit, so the log never disagrees with reality.")

### Try it yourself

Step 3 rejected the pending email. Predict what changes in the trace if you approve it instead.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
# Start a fresh run: the earlier checkpoint was consumed by the rejection.
approved_run = runtime.run(task, "Send a synthetic course update")
print("Paused again on:", approved_run.pending_action["tool"], "|", approved_run.status)

# resume() carries the human answer. The exact arguments come out of the CHECKPOINT,
# not out of a new model call, so the person approved the thing that actually runs.
final = runtime.resume(approved_run.run_id, task, approved=True)
print()
print("Final status :", final.status)
print("Tool output  :", final.output)
print()
print("rejected :", [e["event"] for e in rejected.events[-3:]])
print("approved :", [e["event"] for e in final.events[-4:]])
print("Both record approval_resolved. Only the approved path reaches tool_completed,")
print("and only then does the side effect happen.")

### Checkpoint

**1. Why must the approval answer come back through `resume()` rather than by asking the model again?**

<details><summary>Show answer</summary>

Because the model would be re-deciding, not confirming. The checkpoint holds the exact arguments a human reviewed; a fresh call could produce a different recipient.

</details>

**2. A tool arrives from an outside server carrying `risk="maybe"`. What happens?**

<details><summary>Show answer</summary>

`decide` returns `deny`, because `RISK_POLICY.get(risk, 'deny')` falls back to denial. An unknown label can never be read as permission, and the denial is logged rather than crashing.

</details>

### Recap

- **Limitation seen:** an allow-list cannot express 'visible, but not without a human'.
- **Layer added:** a risk table that fails closed, a checkpointed approval pause, a hard step cap.
- **Evidence:** `send_email` went `cancelled` then `completed`; `erase_workspace` denied; unknown risk denied; 500 steps became 10.